In [48]:
import numpy as np
from collections import OrderedDict

### 합성곱/풀링 계층 구현하기

In [26]:
x = np.random.rand(10,1,28,28)

In [27]:
x.shape

(10, 1, 28, 28)

In [28]:
x[0].shape

(1, 28, 28)

In [29]:
x[9][0]   # 10번째 데이터, 첫 번째 채널

array([[9.32996553e-01, 9.48455424e-01, 4.81316389e-01, 1.36603244e-01,
        2.80894977e-01, 5.55201206e-02, 7.09958920e-02, 8.59267589e-01,
        2.56389894e-01, 9.01792772e-01, 3.10076995e-02, 4.73225735e-01,
        7.37688305e-03, 7.12834498e-01, 8.31578222e-01, 8.29401916e-01,
        3.56003135e-01, 1.72041930e-01, 3.13551793e-01, 5.69882450e-01,
        5.04910698e-01, 6.92749326e-01, 4.87873859e-01, 4.47549630e-02,
        9.23268780e-01, 9.96226966e-01, 6.98662448e-01, 1.67196592e-01],
       [4.35590682e-01, 4.96408786e-01, 2.97818663e-01, 6.17219058e-01,
        9.67963259e-01, 4.65715341e-02, 8.08147090e-01, 1.73060041e-01,
        8.44937762e-01, 2.78430436e-01, 2.00946506e-01, 4.02565000e-01,
        2.27365634e-02, 1.18531558e-01, 2.00750971e-02, 6.58505610e-01,
        3.77370120e-01, 3.92911200e-01, 7.93086599e-01, 3.47405411e-01,
        6.68280229e-01, 2.55329784e-01, 4.91110472e-01, 8.95741742e-02,
        4.18246028e-01, 9.19551937e-01, 9.97870506e-02, 2.13865

### im2col로 데이터 전개하기

In [30]:
# im2col은 입력 데이터(특징 맵)를 필터링(가중치 계산)하기 좋게 전개하는(펼치는) 함수이다. 
# 3차원 입력 데이터에 im2col을 적용하면 2차원 행렬로 바뀐다. 배치 안의 데이터 수까지 포함하여 4차원 데이터를 2차원으로 변환한다.

def im2col(input_data, filter_h, filter_w, stride=1, pad=0): # input_data: (데이터 수, 채널 수, 높이, 너비)의 4차원 배열로 이루어진 입력 데이터, filter_h: 필터의 높이, filter_w: 필터의 너비, stride: 스트라이드, pad: 패딩
    N, C, H, W = input_data.shape               # N: 데이터 수, C: 채널 수, H: 높이, W: 너비
    out_h = (H + 2*pad - filter_h)//stride + 1  # 출력 데이터의 높이 OH, fiter_h는 필터의 높이
    out_w = (W + 2*pad - filter_w)//stride + 1  # 출력 데이터의 너비 OW, filter_w는 필터의 너비

    img = np.pad(input_data, [(0,0), (0,0), (pad, pad), (pad, pad)], 'constant') # 입력 데이터에 pad만큼 0으로 채움
    col = np.zeros((N, C, filter_h, filter_w, out_h, out_w)) # 입력 데이터를 2차원 행렬로 전개할 배열 0으로 초기화

    # 입력 이미지 데이터를 슬라이딩 윈도우 방식으로 필터링하여 2차원 배열로 변환
    # 필터의 높이, 너비만큼 이동하면서 필터를 적용
    for y in range(filter_h):
        y_max = y + stride*out_h
        for x in range(filter_w):
            x_max = x + stride*out_w
            col[:, :, y, x, :, :] = img[:, :, y:y_max:stride, x:x_max:stride]

    col = col.transpose(0, 4, 5, 1, 2, 3).reshape(N*out_h*out_w, -1)
    return col # 2차원 행렬로 전개


In [31]:
x1 = np.random.rand(1, 3, 7, 7) # (데이터 수, 채널 수, 높이, 너비), input_data
col1 = im2col(x1, 5, 5, stride=1, pad=0) # 필터 크기 5x5, 스트라이드 1, 패딩 0
print(col1)

[[6.19818489e-01 8.70748153e-01 1.68558903e-01 2.59309980e-01
  3.98891030e-01 6.59067530e-01 3.24377502e-01 7.10458528e-01
  2.77196113e-01 7.23209400e-01 5.35435974e-01 4.43129965e-01
  5.12330251e-01 2.32309609e-01 3.89960006e-01 5.08517931e-01
  6.68390350e-01 5.44315128e-01 2.23921377e-01 5.36148318e-01
  4.57622225e-01 2.52631504e-01 3.37259881e-01 3.38590113e-01
  6.57720927e-01 1.76485911e-01 8.99988898e-01 6.90851469e-01
  2.80704119e-01 2.40649875e-01 8.47454714e-01 9.15840070e-01
  6.64746559e-01 5.48260779e-01 3.56595664e-01 2.01654295e-01
  4.93814771e-01 7.03375340e-01 5.73171964e-01 2.23681865e-02
  3.57940048e-03 6.66429546e-01 8.44829719e-01 6.00877930e-01
  4.59074833e-01 2.27704276e-02 5.18791282e-01 1.23553261e-01
  4.18649744e-01 9.13671813e-02 7.99211528e-01 4.70143863e-01
  3.73882823e-01 8.68965411e-01 9.94446811e-01 5.76624176e-02
  5.60455078e-01 7.10538319e-01 5.69044773e-01 3.09708637e-01
  4.72287866e-01 7.63793904e-01 4.42606769e-01 7.29968128e-01
  4.5441

### 합성곱 계층 구현  

In [32]:
class Convolution:
  def __init__(self,W,b,stride=1,pad=0):
    self.W = W            # 필터(가중치)
    self.b = b            # 편향
    self.stride = stride  # 스트라이드
    self.pad = pad        # 패딩
  
  def forward(self,x):
    FN, C, FH, FW = self.W.shape  # 필터, FN: 필터 개수, C: 채널 수, FH: 필터 높이, FW: 필터 너비
    N, C, H, W = x.shape          # 입력 데이터, N: 데이터 수, C: 채널 수, H: 높이, W: 너비
    out_h = 1 + int((H + 2*self.pad - FH) / self.stride) # 출력 데이터 높이, OH
    out_w = 1 + int((W + 2*self.pad - FW) / self.stride) # 출력 데이터 너비, OW

    col = im2col(x, FH, FW, self.stride, self.pad) # 입력 데이터를 2차원 배열로 전개한 col
    col_W = self.W.reshape(FN, -1).T               # 필터를 reshape를 사용하여 2차원 배열로 전개한 col_W, T는 transpose를 의미

    out = np.dot(col, col_W) + self.b              # 합성곱 연산 수행 (행렬의 내적)
    out = out.reshape(N, out_h, out_w, -1).transpose(0, 3, 1, 2)  # 출력 데이터를 reshape하여 transpose, 형상을 (N(0), H(1), W(2), C(3))에서 (N(0), C(3), H(1), W(2))로 변경

    return out

In [33]:
x1 = np.random.rand(1, 3, 7, 7) # (데이터 수, 채널 수, 높이, 너비), input_data
col1 = im2col(x1, 5, 5, stride=1, pad=0) # 필터 크기 5x5, 스트라이드 1, 패딩 0
print(col1)                              # input_data를 2차원 배열로 전개

# Convolution 클래스 사용
W = np.random.rand(3, 3, 5, 5) # (필터 수, 채널 수, 높이, 너비), filter 5x5x3
b = np.random.rand(1)          # 편향
conv1 = Convolution(W, b, stride=1, pad=0) # Convolution 클래스 생성

out1 = conv1.forward(x1) # 순전파
print(out1)              # iput_data에 필터를 적용한 결과 출력

[[0.58595963 0.93044062 0.13520508 0.82345341 0.99737324 0.02839692
  0.94144054 0.0998631  0.38654968 0.72256658 0.82287926 0.29303176
  0.90874182 0.22307241 0.18609281 0.76307245 0.85496363 0.4685354
  0.51280669 0.0514145  0.06134306 0.214902   0.14823271 0.52219427
  0.21818467 0.49283505 0.68294459 0.27609363 0.33514817 0.85242937
  0.91748076 0.55227754 0.26281157 0.7731887  0.58374347 0.65409177
  0.71845216 0.90539514 0.12558903 0.70463105 0.5049862  0.4630031
  0.51782196 0.57477328 0.18912621 0.02619133 0.67511957 0.46304877
  0.14200944 0.27868343 0.98915933 0.49464324 0.24872971 0.19944806
  0.40999172 0.57981045 0.24424321 0.50093562 0.39721079 0.0345694
  0.11070585 0.04353588 0.49259016 0.65102391 0.93078005 0.83765732
  0.22247295 0.31409423 0.59054816 0.41486253 0.20891213 0.96038543
  0.58545095 0.41600582 0.86598836]
 [0.93044062 0.13520508 0.82345341 0.99737324 0.97823539 0.94144054
  0.0998631  0.38654968 0.72256658 0.56152567 0.29303176 0.90874182
  0.22307241 0.

### Pooling 계층  

In [45]:
class Pooling:
  def __init__(self,pool_h,pool_w,stride=1,pad=0):
    self.pool_h = pool_h  # 풀링 높이
    self.pool_w = pool_w  # 풀링 너비
    self.stride = stride  # 스트라이드
    self.pad = pad        # 패딩
    
  def forward(self,x):
    N, C, H, W = x.shape
    print(x.shape)
    out_h = int(1 + (H - self.pool_h) / self.stride) # 출력 데이터 높이 OH
    out_w = int(1 + (W - self.pool_w) / self.stride) # 출력 데이터 너비 OW
    
    # 전개(1)
    col = im2col(x, self.pool_h, self.pool_w, self.stride, self.pad)
    print(col)
    col = col.reshape(-1, self.pool_h*self.pool_w)
    print(col)
    
    
    # 최댓값(2)
    out = np.max(col, axis=1) # Max pooling, 각 행마다 최댓값을 구함
    
    # 성형(3)
    out = out.reshape(N, out_h, out_w, C).transpose(0, 3, 1, 2)
    
    return out

In [ ]:
# out1(합성곱 층 결과) 을 Pooling 클래스의 forward 메서드에 넣어 풀링 연산을 수행

pool1 = Pooling(pool_h=2, pool_w=2, stride=1) # Pooling 클래스 생성, 풀링 높이 2, 풀링 너비 2, 스트라이드 1
out2 = pool1.forward(out1) # Pooling 순전파
print(out2)                # out1에 풀링을 적용한 결과 출력

### CNN 구현하기   

### 필요한 계층 불러오기  

In [55]:
# 순전파와 역전파를 포함한 ReLU 계층 구현

class Relu: 
  def __init__(self):
    self.mask = None # mask는 True/False로 구성된 넘파이 배열, 순전파의 입력인 x의 원소 값이 0 이하인 인덱스는 True, 그 외(0보다 큰 원소)는 False로 유지
  
  # 순전파
  def forward(self,x):
    self.mask = (x <= 0) # x의 원소 값이 0 이하인 인덱스를 True로 설정
    out = x.copy() # 입력 x를 복사하여 out에 저장
    out[self.mask] = 0 # mask의 원소가 True인 인덱스에 대응하는 원소를 0으로 설정, 즉 x가 0보다 작으면 0으로 변환, 그 외는 그대로 유지
    
    return out
  
  # 역전파
  def backward(self,dout):
    dout[self.mask] = 0
    dx = dout # 순전파 때의 입력인 x가 0보다 작으면 역전파 때의 값은 0, 그 외는 상류 값을 그대로 전달한다. 
    
    return dx

In [56]:
# 신경망의 순전파 때 수행하는 행렬의 내적을 기하학에서는 어파인 변환(Affine Transformation)이라고 불러서 Affine 계층이라는 이름을 사용한다.
# Affine 계층은 가중치 신호의 총합을 계산하고, 편향을 더한다.

# N개의 데이터를 묶어 순전파와 역전파를 수행한 Affine 계층 구현
class Affine: 
  def __init__(self,w,b):
    self.w = w  # 가중치
    self.b = b  # 편향
    self.x = None # 입력
    self.dw = None # 가중치의 미분
    self.db = None # 편향의 미분
   
  # 순전파  
  def forward(self, x):
    self.x = x
    out = np.dot(x,self.w) + self.b  # y = WX + B
    
    return out
   
  # 역전파  
  def backward(self,dout):
    dx = np.dot(dout,self.w.T)  # dx = dL/dy * W^T, self.w.T는 self.w의 전치행렬, 입력 x에 대한 상류 값의 미분으로 이전 층으로 전파해야 할 값이다
    self.dw = np.dot(self.x.T,dout) # dw = X^T * dL/dy, self.x.T는 self.x의 전치행렬
    self.db = np.sum(dout, axis = 0) # db = dL/dy의 각 원소의 총합이다, axis = 0은 행 방향으로 더한다
    
    return dx

In [58]:
def softmax(x):
    if x.ndim == 2:
        x = x - np.max(x, axis=1, keepdims=True) # 오버플로 대책
        x = np.exp(x)
        x /= np.sum(x, axis=1, keepdims=True)
    elif x.ndim == 1:
        x = x - np.max(x) # 오버플로 대책
        x = np.exp(x) / np.sum(np.exp(x))
    return x
  
  # 평균 손실함수
def cross_entropy_error(y, t):
    if y.ndim == 1:
        t = t.reshape(1, t.size) # t는 실제 정답 레이블
        y = y.reshape(1, y.size) # y는 신경망의 출력

    batch_size = y.shape[0]
    return -np.sum(t * np.log(y + 1e-7)) / batch_size  # np.log(0)이 -inf가 되는 것을 방지하기 위해 아주 작은 delta(1e-7)를 더해줌

# Softmax-with-Loss 계층

class SoftmaxWithLoss:
  def __init__(self):
    self.loss = None # 손실
    self.y = None # softmax의 출력
    self.t = None # 정답 레이블(원-핫 인코딩 형태)
    
  def forward(self,x,t):
    self.t = t
    self.y = softmax(x)
    self.loss = cross_entropy_error(self.y,self.t)
    
    # 순전파의 결과인 손실을 반환한다
    return self.loss
  
  def backward(self,dout=1):
    batch_size = self.t.shape[0]
    dx = (self.y - self.t) / batch_size # 역전파의 결과로는 데이터의 개수로 나눠서 데이터 1개당 오차를 앞 계층으로 전파한다
    
    return dx

In [57]:
# {Convolution-ReLU-Pooling}-{Affine-ReLU}-{Affine-Softmax} 구현

class SimpleConvNet:
  def __init__(self,input_dim=(1,28,28),                                        # 입력 데이터(채널 수, 높이, 너비) 1x28x28
               conv_param={'filter_num':30,'filter_size':5,'pad':0,'stride':1}, # 합성곱 계층의 하이퍼파라미터(필터 개수(FN), 필터 크기(FH,FW), 패딩, 스트라이드)
               hidden_size=100,output_size=10,weight_init_std=0.01):            # 은닉층 뉴런 수, 출력층의 뉴런 수(클래스 수), 가중치 초기화 표준편차
    filter_num = conv_param['filter_num']   # 필터 개수, FN
    filter_size = conv_param['filter_size'] # 필터 크기, FH, FW
    filter_pad = conv_param['pad']          # 패딩
    filter_stride = conv_param['stride']    # 스트라이드
    input_size = input_dim[1]               # 입력 데이터(채널 수, 높이, 너비)의 높이
    conv_output_size = (input_size - filter_size + 2*filter_pad) / filter_stride + 1 # 합성곱 계층 출력 크기 30x24x24
    pool_output_size = int(filter_num * (conv_output_size/2) * (conv_output_size/2)) # 풀링 계층 출력 크기 30x12x12
    
    # 1층의 합성곱 계층과 2, 3층의 완전연결 계층의 가중치와 편향 생성
    self.params = {}
    # 합성곱 계층의 가중치와 편향
    self.params['W1'] = weight_init_std * np.random.randn(filter_num, input_dim[0], filter_size, filter_size)
    self.params['b1'] = np.zeros(filter_num)
    
    # 2층 완전연결 계층의 가중치와 편향
    self.params['W2'] = weight_init_std * np.random.randn(pool_output_size, hidden_size)
    self.params['b2'] = np.zeros(hidden_size)
    
    # 3층 완전연결 계층의 가중치와 편향
    self.params['W3'] = weight_init_std * np.random.randn(hidden_size, output_size)
    self.params['b3'] = np.zeros(output_size)
    
    # CNN 구성 계층 생성
    self.layers = OrderedDict()
    # 1층 합성곱 계층
    self.layers['Conv1'] = Convolution(self.params['W1'], self.params['b1'], conv_param['stride'], conv_param['pad']) # 필터(가중치), 편향, 스트라이드, 패딩
    self.layers['Relu1'] = Relu()                                # 활성화 함수 ReLU
    self.layers['Pool1'] = Pooling(pool_h=2, pool_w=2, stride=2) # 풀링층, 2x2 Max pooling, 스트라이드 2
    
    # 2층 완전연결 계층
    self.layers['Affine1'] = Affine(self.params['W2'], self.params['b2'])
    self.layers['Relu2'] = Relu()                                # 활성화 함수 ReLU
    
    # 3층 완전연결 계층
    self.layers['Affine2'] = Affine(self.params['W3'], self.params['b3'])
    self.last_layer = SoftmaxWithLoss()                          # 출력층 Softmax-with-Loss 계층